# 02 · Cleaning + simulated OBD-II features

Applies the cleaning contract and synthesises `battery_soh` and `trans_adapt_offset` (see spec). Logic lives in `ml.ingest` / `ml.features` — this notebook narrates and visualises it.

In [ ]:
import sys; sys.path.insert(0, '..' if __import__('pathlib').Path('..').joinpath('app').exists() else '../..')
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from ml import ingest
sns.set_theme()
raw = ingest.load_raw(); df = ingest.build_engineered_csv()
print('raw:', raw.shape, '-> engineered:', df.shape); df.head()

### `battery_soh` — Starter Battery State of Health
Time-dominant for petrol/diesel; non-linear in mileage for hybrids.

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(11,4))
df['battery_soh'].plot.hist(bins=40, ax=ax[0], title='battery_soh (%)')
sns.scatterplot(data=df.sample(min(2000,len(df)), random_state=0), x='mileage', y='battery_soh',
                hue='fuel_type', s=10, ax=ax[1]); ax[1].set_title('SoH vs mileage'); plt.tight_layout()

### `trans_adapt_offset` — Transmission Adaptation Offset
Manual is exactly 0 (own branch); automatics are strictly negative and scale with mileage.

In [ ]:
print(df.groupby('transmission')['trans_adapt_offset'].describe()[['mean','min','max']])
auto = df[df['transmission'] != 'Manual'].sample(min(2000,len(df)), random_state=0)
sns.scatterplot(data=auto, x='mileage', y='trans_adapt_offset', s=10)
plt.title('offset vs mileage (non-manual)')

Engineered dataset written to `data/merc_engineered.csv` — this is what the model trains on.